# Conteo de porotos — pipeline completo en Colab (GPU)

Corre **descarga -> ETAPA 1 -> ETAPA 2 -> entrenamiento YOLO** y te deja `best.pt`.

Primero: **Entorno de ejecucion -> Cambiar tipo de entorno -> GPU (T4)**.

In [ ]:
!nvidia-smi -L  # verificar GPU

## 1 - Subir el proyecto (zip de codigo, liviano)

Subi **`poroto_cv_colab.zip`** (solo codigo, unos KB). Tambien sirve el zip completo.

In [ ]:
from google.colab import files
import zipfile, os, glob
up = files.upload()                 # elegi el zip
z = next(k for k in up if k.endswith('.zip'))
with zipfile.ZipFile(z) as f: f.extractall('proj')
# ubicar la carpeta del proyecto (la que contiene synthetic_data)
root = os.path.dirname(glob.glob('proj/**/synthetic_data', recursive=True)[0])
os.chdir(root); print('cwd:', os.getcwd())

In [ ]:
# Dependencias (Colab ya trae torch)
!pip install -q ultralytics kagglehub
!pip install -q -r requirements.txt

## 2 - Credenciales de Kaggle (con secretos de Colab, sin subir archivos)

**Opcion facil (recomendada):**
1. Copia el valor de tu token (API Tokens -> Generate New Token). Si no lo copiaste, genera otro y copialo ahi mismo.
2. En Colab, barra izquierda, icono de **llave** (Secrets) -> **+ Agregar secreto** -> Nombre `KAGGLE_API_TOKEN`, Valor: pega el token -> activa **Acceso del notebook**.
3. Corre la celda de abajo.

**Alternativa (kaggle.json clasico):** en Kaggle baja hasta **Legacy API Credentials -> Create Legacy API Key** (eso si descarga kaggle.json). Si la celda no encuentra secretos, te lo va a pedir para subir.

In [ ]:
import os
def setup_kaggle():
    try:
        from google.colab import userdata
        tok = userdata.get('KAGGLE_API_TOKEN')
        if tok:
            os.makedirs('/root/.kaggle', exist_ok=True)
            open('/root/.kaggle/access_token','w').write(tok.strip())
            os.chmod('/root/.kaggle/access_token', 0o600)
            os.environ['KAGGLE_API_TOKEN'] = tok.strip()
            print('OK -> KAGGLE_API_TOKEN'); return
    except Exception as e: print('sin KAGGLE_API_TOKEN:', e)
    try:
        from google.colab import userdata
        u, k = userdata.get('KAGGLE_USERNAME'), userdata.get('KAGGLE_KEY')
        if u and k:
            os.environ['KAGGLE_USERNAME']=u; os.environ['KAGGLE_KEY']=k
            print('OK -> KAGGLE_USERNAME/KAGGLE_KEY'); return
    except Exception: pass
    from google.colab import files; import shutil
    print('No hay secretos; subi kaggle.json (Legacy API Credentials)')
    files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.move('kaggle.json','/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
setup_kaggle()

## 3 - Correr el pipeline completo

En T4, `yolov8s` con 500 imagenes y 80 epocas entra comodo. Bajalo si tenes poco tiempo.

In [ ]:
%env N_IMAGES=500
%env EPOCHS=80
%env MODEL=yolov8s.pt
%env IMGSZ=1024
%env LIMIT_GRAINS=400
!bash run_pipeline.sh

## 4 - Descargar los pesos entrenados

In [ ]:
from google.colab import files
files.download('weights/best.pt')

## 5 - (Opcional) Probar la inferencia con el modelo real

In [ ]:
import glob, cv2
from matplotlib import pyplot as plt
from inference.model import YOLODetector
from inference.counter import count_and_annotate
det = YOLODetector('weights/best.pt', conf=0.25)
p = sorted(glob.glob('data/synthetic/images/val/*.jpg'))[0]
bgr = cv2.imread(p)
res = count_and_annotate(bgr, det.predict(bgr), 'colab', det.version)
print('porotos:', res.cantidad, '| conf:', round(res.avg_confidence,3), '| success:', res.success)
plt.figure(figsize=(8,8)); plt.imshow(cv2.cvtColor(res.annotated, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()